
# Credit Risk Prediction — End-to-End Machine Learning Project

This notebook builds a complete credit-risk classification workflow using the **German Credit** dataset.

It covers:

1. Data loading and inspection  
2. Data cleaning  
3. Exploratory data analysis  
4. Encoding categorical variables  
5. Train/test split  
6. Baseline and tree-based models  
7. Model evaluation  
8. Feature importance  
9. Final model selection  
10. Exporting the trained model and encoders for a Streamlit app  

> **Dataset expected:** a CSV such as `german_credit_data.csv` containing columns similar to:
> `Age`, `Sex`, `Job`, `Housing`, `Saving accounts`, `Checking account`,
> `Credit amount`, `Duration`, and `Risk`.
>
> If the local CSV is not found, the notebook attempts to download the German Credit dataset from OpenML.


In [1]:

# Core libraries
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance

import joblib

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42



## 1. Load the dataset

The loader first checks for a local CSV. If none is found, it tries OpenML.

If your file has a different name, change `LOCAL_DATA_PATH`.


In [3]:

LOCAL_DATA_PATH = Path("german_credit_data.csv")

if LOCAL_DATA_PATH.exists():
    df = pd.read_csv(LOCAL_DATA_PATH)
    print(f"Loaded local dataset: {LOCAL_DATA_PATH}")
else:
    print("Local CSV not found. Attempting to load German Credit data from OpenML...")
    from sklearn.datasets import fetch_openml

    credit = fetch_openml(name="credit-g", version=1, as_frame=True)
    df = credit.frame.copy()

    # Rename OpenML columns into the simpler names used by this project.
    rename_map = {
        "checking_status": "Checking account",
        "duration": "Duration",
        "credit_amount": "Credit amount",
        "personal_status": "Sex",
        "age": "Age",
        "employment": "Job",
        "housing": "Housing",
        "savings_status": "Saving accounts",
        "class": "Risk",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

print(df.shape)
df.head()


Local CSV not found. Attempting to load German Credit data from OpenML...
(1000, 21)


,Checking account,Duration,credit_history,purpose,Credit amount,Saving accounts,Job,installment_commitment,Sex,other_parties,residence_since,property_magnitude,Age,other_payment_plans,Housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,Risk
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,4,real estate,67,none,own,2,skilled,1,yes,yes,good
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,2,real estate,22,none,own,1,skilled,1,none,yes,bad
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,3,real estate,49,none,own,1,unskilled resident,2,none,yes,good
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,4,life insurance,45,none,for free,1,skilled,2,none,yes,good
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,4,no known property,53,none,for free,2,skilled,2,none,yes,bad


## 2. Initial inspection

In [ ]:

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nInfo:")
df.info()


In [ ]:

df.describe(include="all").T


## 3. Clean the data

In [ ]:

# Remove common unnamed index columns created when exporting CSV files.
unnamed_cols = [c for c in df.columns if str(c).lower().startswith("unnamed")]
if unnamed_cols:
    df = df.drop(columns=unnamed_cols)

# Standardise column names by stripping accidental whitespace.
df.columns = [str(c).strip() for c in df.columns]

print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]


In [ ]:

# Fill missing categorical values with an explicit "unknown" category.
categorical_candidates = [
    "Sex",
    "Housing",
    "Saving accounts",
    "Checking account",
]

for col in categorical_candidates:
    if col in df.columns:
        df[col] = df[col].astype("object").fillna("unknown")

# Convert key numeric columns safely.
numeric_candidates = ["Age", "Job", "Credit amount", "Duration"]
for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())

df.head()


## 4. Exploratory data analysis

In [ ]:

target_col = "Risk"

if target_col not in df.columns:
    raise ValueError(
        f"Expected a target column called '{target_col}'. "
        f"Available columns: {df.columns.tolist()}"
    )

df[target_col].value_counts(dropna=False)


In [ ]:

# Target distribution
df[target_col].value_counts().plot(kind="bar")
plt.title("Credit Risk Distribution")
plt.xlabel("Risk")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()


In [ ]:

# Numeric distributions
numeric_cols = [c for c in ["Age", "Job", "Credit amount", "Duration"] if c in df.columns]

for col in numeric_cols:
    plt.figure(figsize=(6, 4))
    df[col].plot(kind="hist", bins=25)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()


In [ ]:

# Compare selected numeric variables by target class.
for col in [c for c in ["Age", "Credit amount", "Duration"] if c in df.columns]:
    grouped = [group[col].dropna().values for _, group in df.groupby(target_col)]
    labels = [str(name) for name, _ in df.groupby(target_col)]

    plt.figure(figsize=(6, 4))
    plt.boxplot(grouped)
    plt.xticks(range(1, len(labels) + 1), labels)
    plt.title(f"{col} by Credit Risk")
    plt.xlabel("Risk")
    plt.ylabel(col)
    plt.show()


In [ ]:

# Categorical distributions
for col in [c for c in categorical_candidates if c in df.columns]:
    counts = pd.crosstab(df[col], df[target_col], normalize="index")
    counts.plot(kind="bar", figsize=(7, 4))
    plt.title(f"{target_col} by {col}")
    plt.ylabel("Proportion")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()



## 5. Select modelling features

The Streamlit app uses the following applicant-level variables:

- Age
- Sex
- Job
- Housing
- Saving accounts
- Checking account
- Credit amount
- Duration

The notebook therefore trains the deployment model on the same fields.


In [ ]:

feature_cols = [
    "Age",
    "Sex",
    "Job",
    "Housing",
    "Saving accounts",
    "Checking account",
    "Credit amount",
    "Duration",
]

missing_features = [c for c in feature_cols if c not in df.columns]
if missing_features:
    raise ValueError(
        "The dataset is missing columns required by the Streamlit app: "
        + ", ".join(missing_features)
    )

model_df = df[feature_cols + [target_col]].copy()
model_df.head()



## 6. Encode categorical variables

Separate `LabelEncoder` objects are saved for each categorical input so that the
Streamlit application can transform user-entered categories in exactly the same way
as the training notebook.


In [ ]:

categorical_cols = [
    "Sex",
    "Housing",
    "Saving accounts",
    "Checking account",
]

encoders = {}

for col in categorical_cols:
    encoder = LabelEncoder()
    model_df[col] = encoder.fit_transform(model_df[col].astype(str))
    encoders[col] = encoder

target_encoder = LabelEncoder()
model_df[target_col] = target_encoder.fit_transform(model_df[target_col].astype(str))

print("Target mapping:")
for cls, encoded in zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)):
    print(f"{cls!r} -> {encoded}")

model_df.head()


## 7. Train/test split

In [ ]:

X = model_df[feature_cols]
y = model_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).sort_index())


## 8. Train several models

In [ ]:

models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        ]
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    ),
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    accuracy = accuracy_score(y_test, preds)

    if hasattr(model, "predict_proba") and len(np.unique(y)) == 2:
        probs = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, probs)
    else:
        auc = np.nan

    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy",
        n_jobs=-1,
    )

    results.append({
        "Model": name,
        "Test Accuracy": accuracy,
        "ROC AUC": auc,
        "CV Accuracy Mean": cv_scores.mean(),
        "CV Accuracy Std": cv_scores.std(),
    })

results_df = pd.DataFrame(results).sort_values(
    by=["ROC AUC", "Test Accuracy"],
    ascending=False,
    na_position="last",
).reset_index(drop=True)

results_df


## 9. Evaluate the Extra Trees model

In [ ]:

extra_trees = models["Extra Trees"]

y_pred = extra_trees.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[str(c) for c in target_encoder.classes_]
    )
)


In [ ]:

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_encoder.classes_
)
disp.plot()
plt.title("Extra Trees — Confusion Matrix")
plt.show()


In [ ]:

if len(target_encoder.classes_) == 2:
    y_prob = extra_trees.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    print(f"ROC AUC: {auc:.3f}")

    RocCurveDisplay.from_predictions(y_test, y_prob)
    plt.title("Extra Trees — ROC Curve")
    plt.show()


## 10. Feature importance

In [ ]:

feature_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": extra_trees.feature_importances_,
}).sort_values("Importance", ascending=True)

plt.figure(figsize=(7, 5))
plt.barh(feature_importance["Feature"], feature_importance["Importance"])
plt.title("Extra Trees Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

feature_importance.sort_values("Importance", ascending=False)


In [ ]:

# Permutation importance provides a model-agnostic check.
perm = permutation_importance(
    extra_trees,
    X_test,
    y_test,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="accuracy",
)

permutation_df = pd.DataFrame({
    "Feature": feature_cols,
    "Permutation Importance": perm.importances_mean,
}).sort_values("Permutation Importance", ascending=False)

permutation_df



## 11. Retrain the final model on all available data

After model comparison and evaluation, retrain the selected Extra Trees classifier on
the full modelling dataset before exporting it for deployment.


In [ ]:

final_model = ExtraTreesClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

final_model.fit(X, y)

print("Final model trained on", len(X), "rows.")


## 12. Save model and encoders

In [ ]:

ARTIFACT_DIR = Path(".")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, ARTIFACT_DIR / "extra_trees_credit_model.pkl")

for col, encoder in encoders.items():
    joblib.dump(encoder, ARTIFACT_DIR / f"{col}_encoder.pkl")

joblib.dump(target_encoder, ARTIFACT_DIR / "target_encoder.pkl")

print("Saved:")
print("- extra_trees_credit_model.pkl")
for col in categorical_cols:
    print(f"- {col}_encoder.pkl")
print("- target_encoder.pkl")


## 13. Verify the exported artifacts

In [ ]:

loaded_model = joblib.load("extra_trees_credit_model.pkl")
loaded_target_encoder = joblib.load("target_encoder.pkl")

loaded_encoders = {
    col: joblib.load(f"{col}_encoder.pkl")
    for col in categorical_cols
}

print("Model:", type(loaded_model).__name__)
print("Target classes:", loaded_target_encoder.classes_)

for col, enc in loaded_encoders.items():
    print(f"{col}: {list(enc.classes_)}")


## 14. Test a single prediction

In [ ]:

# Replace these values with a realistic applicant.
example_applicant = {
    "Age": 30,
    "Sex": "male",
    "Job": 1,
    "Housing": "own",
    "Saving accounts": "little",
    "Checking account": "little",
    "Credit amount": 1000,
    "Duration": 12,
}

encoded_applicant = example_applicant.copy()

for col in categorical_cols:
    encoder = loaded_encoders[col]
    value = str(encoded_applicant[col])

    if value not in encoder.classes_:
        raise ValueError(
            f"Unknown value {value!r} for {col}. "
            f"Allowed values: {list(encoder.classes_)}"
        )

    encoded_applicant[col] = encoder.transform([value])[0]

example_df = pd.DataFrame([encoded_applicant], columns=feature_cols)

prediction_encoded = loaded_model.predict(example_df)[0]
prediction_label = loaded_target_encoder.inverse_transform([prediction_encoded])[0]

print("Predicted credit risk:", prediction_label)



## 16. Suggested next improvements

Potential extensions:

- Hyperparameter tuning with `GridSearchCV` or `RandomizedSearchCV`
- Calibration of predicted probabilities
- Class-imbalance analysis
- SHAP-based model explanations
- FastAPI endpoint for model inference
- `pytest` tests for preprocessing and predictions
- GitHub Actions CI
- Streamlit Community Cloud deployment
